In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.vectorstores import VectorStore
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from dotenv import load_dotenv
import os
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI
os.environ["google_api_key"] = os.getenv("google_api_key")
llm = ChatGoogleGenerativeAI(model = "gemini-1.5-pro")

# os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# llm = ChatGroq(model = "llama3-8b-8192")

loader = Docx2txtLoader("eda_report.docx")

data = loader.load()

vectorstore = InMemoryVectorStore.from_documents(documents=data,embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))
retriever = vectorstore.as_retriever()

from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain

# Define the system prompt for the LLM
system_prompt = (
    """You are a highly knowledgeable data scientist, capable of answering any questions in the field of AI, including those related to reports provided by the user.
    Only respond to questions strictly related to AI.
    Ensure all answers are concise, accurate, and explicit.
    Dont be concise with your answers"""
    "\n\n"
    "{context}"
)

# Define the prompt template for the conversation
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# Create the question-answer chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)

# Create the retrieval chain with the retriever and question-answer chain
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

while True:
    input_data = {
        "input": input("You: ")
    }
    
    if input_data == "exit":
        break
    response = rag_chain.invoke(input_data)
    print("Bot:", response['answer'])

    if response['answer'] == "Goodbye!":
        break

In [5]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.vectorstores import VectorStore
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain
from langchain.memory import ConversationBufferMemory
import os


class AIChatAssistant:
    def __init__(self, doc_path, llm_model="gemini-1.5-pro", embedding_model="all-MiniLM-L6-v2"):
        load_dotenv()

        # Load environment variables
        os.environ["google_api_key"] = os.getenv("google_api_key")

        # Initialize LLM
        self.llm = ChatGoogleGenerativeAI(model=llm_model)

        # Load document
        self.loader = Docx2txtLoader(doc_path)
        self.data = self.loader.load()

        # Create vector store
        self.vectorstore = InMemoryVectorStore.from_documents(
            documents=self.data, embedding=HuggingFaceEmbeddings(model_name=embedding_model)
        )
        self.retriever = self.vectorstore.as_retriever()

        # Initialize memory
        self.memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

        # Define prompts
        self.system_prompt = (
            """You are a highly knowledgeable data scientist, capable of answering any questions in the field of AI, including those related to reports provided by the user.
            Dont respond to any other questions about any other field than AI.
            Ensure all answers are concise, accurate, and explicit.
            Don't be concise with your answers."""
            "\n\n"
            "{context}\n\n"
            "Chat history:\n{chat_history}"
        )

        self.prompt = ChatPromptTemplate.from_messages(
            [
                ("system", self.system_prompt),
                ("human", "{input}"),
            ]
        )

        # Create chains
        self.question_answer_chain = create_stuff_documents_chain(self.llm, self.prompt)
        self.rag_chain = create_retrieval_chain(self.retriever, self.question_answer_chain)

    def ask_question(self, question):
        # Add memory context
        input_data = {
            "input": question,
            "chat_history": self.memory.load_memory_variables({})["chat_history"],
        }

        response = self.rag_chain.invoke(input_data)
        # Update memory
        self.memory.save_context({"input": question}, {"output": response["answer"]})
        return response["answer"]

    def run_chat(self):
        detailed_summary = self.ask_question("provide detailed information about the report, machine learning models that can be used as well as the data cleaning technique that can be used")
        print("Bot:", detailed_summary)
        print("bot: {ask_questions('')}")
        print()
        while True:
            user_input = input("You: ")
            
            if user_input.lower() == "exit":
                print("Goodbye!")
                break

            answer = self.ask_question(user_input)
            print(f"you: {user_input}")
            print("Bot:", answer)
            print("****************************************************************************************************************")


# Example usage
if __name__ == "__main__":
    assistant = AIChatAssistant(doc_path="eda_report.docx")
    assistant.run_chat()


Bot: The report describes a dataset related to CO2 emissions. It contains 5,677 rows and 5 columns, with no missing values.  Let's break down the columns and potential uses:

* **Country (object):**  Categorical data representing the country of origin for the emission data.  With 190 unique values, this column is suitable for grouping and comparison across countries.

* **Region (object):** Categorical data representing the region. This likely groups countries into larger geographical areas. With only 5 unique values, this offers a higher-level analysis than 'Country'.  It can be used for aggregated analysis or potentially as a feature in some models.

* **Date (object):** Currently stored as an object, meaning it's treated as text. This needs to be converted to a proper datetime format for time series analysis. The 30 unique values suggest data may be monthly or daily across a specific time span.  This is crucial for understanding trends and seasonality.

* **Kilotons of CO2 (float64)

In [17]:
import pandas as pd
from docx import Document
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT
from docx.shared import Pt
import os

def generate_eda_report(csv_file_path, output_docx_path="eda_report.docx"):
    # Check if file exists
    if not os.path.exists(csv_file_path):
        print(f"File '{csv_file_path}' not found.")
        return

    # Load the CSV file into a Pandas DataFrame
    df = pd.read_csv(csv_file_path)
    
    # Remove completely empty columns
    df.dropna(axis=1, how='all', inplace=True)
    
    # Create a Word document
    doc = Document()
    
    # Add title
    title = doc.add_heading("Exploratory Data Analysis Report", level=1)
    title.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

    # Add dataset summary
    doc.add_heading("1. Dataset Summary", level=2)
    doc.add_paragraph(f"Number of Rows: {df.shape[0]}")
    doc.add_paragraph(f"Number of Columns: {df.shape[1]}")
    
    # Column-wise information
    doc.add_heading("2. Column Details", level=2)
    column_summary = df.describe(include="all").transpose()
    for column, details in column_summary.iterrows():
        doc.add_paragraph(f"Column: {column}")
        doc.add_paragraph(f"  Data Type: {df[column].dtype}")
        doc.add_paragraph(f"  Non-Null Count: {details.get('count', 'N/A')}")
        if pd.api.types.is_numeric_dtype(df[column]):
            doc.add_paragraph(f"  Mean: {details.get('mean', 'N/A'):.2f}")
            doc.add_paragraph(f"  Standard Deviation: {details.get('std', 'N/A'):.2f}")
            doc.add_paragraph(f"  Minimum: {details.get('min', 'N/A'):.2f}")
            doc.add_paragraph(f"  Maximum: {details.get('max', 'N/A'):.2f}")
        else:
            doc.add_paragraph(f"  Unique Values: {details.get('unique', 'N/A')}")
        doc.add_paragraph("\n")

    # Missing values analysis
    doc.add_heading("3. Missing Values Analysis", level=2)
    missing_values = df.isnull().sum()
    total_missing = missing_values.sum()
    doc.add_paragraph(f"Total Missing Values: {total_missing}")
    if total_missing > 0:
        doc.add_paragraph("Columns with Missing Values:")
        for col, miss_count in missing_values[missing_values > 0].items():
            doc.add_paragraph(f"  {col}: {miss_count} missing values")

    # Correlation matrix (only for numeric columns that can be converted to numbers)
    numeric_df = df.select_dtypes(include=["number"])

    # Try to convert non-numeric columns to numeric if possible (e.g., strings like '100' become 100)
    for column in numeric_df.columns:
        numeric_df[column] = pd.to_numeric(numeric_df[column], errors='coerce')
    
    if numeric_df.shape[1] > 1:
        doc.add_heading("4. Correlation Analysis", level=2)
        correlation_matrix = numeric_df.corr()
        doc.add_paragraph("Pairwise correlation matrix:")
        doc.add_paragraph(correlation_matrix.to_string())
    else:
        doc.add_paragraph("No numeric columns available for correlation analysis.")

    # Save the Word document
    doc.save(output_docx_path)
    print(f"EDA report saved as: {output_docx_path}")

# Example usage
csv_file_path = "carbon.csv"  # Replace with your CSV file path
generate_eda_report(csv_file_path, "eda_report.docx")


EDA report saved as: eda_report.docx


In [47]:
import pandas as pd
from docx import Document
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT
import os

def generate_eda_report(csv_file_path, output_docx_path="eda_report.docx"):
    # Check if file exists
    if not os.path.exists(csv_file_path):
        print(f"File '{csv_file_path}' not found.")
        return

    try:
        # Load the CSV file into a Pandas DataFrame
        df = pd.read_csv(csv_file_path)
        
        # Remove completely empty columns
        df.dropna(axis=1, how='all', inplace=True)
    except Exception as e:
        print(f"Error loading the file or cleaning data: {e}")
        return

    # Create a Word document
    doc = Document()
    
    # Add title
    title = doc.add_heading("Exploratory Data Analysis Report", level=1)
    title.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

    try:
        # Add dataset summary
        doc.add_heading("1. Dataset Summary", level=2)
        doc.add_paragraph(f"Number of Rows: {df.shape[0]}")
        doc.add_paragraph(f"Number of Columns: {df.shape[1]}")
    except Exception as e:
        doc.add_paragraph(f"Error generating dataset summary: {e}")

    try:
        # Column-wise information
        doc.add_heading("2. Column Details", level=2)
        column_summary = df.describe(include="all").transpose()
        for column, details in column_summary.iterrows():
            doc.add_paragraph(f"Column: {column}")
            doc.add_paragraph(f"  Data Type: {df[column].dtype}")
            doc.add_paragraph(f"  Non-Null Count: {details.get('count', 'N/A')}")
            if pd.api.types.is_numeric_dtype(df[column]):
                doc.add_paragraph(f"  Mean: {details.get('mean', 'N/A'):.2f}")
                doc.add_paragraph(f"  Standard Deviation: {details.get('std', 'N/A'):.2f}")
                doc.add_paragraph(f"  Minimum: {details.get('min', 'N/A'):.2f}")
                doc.add_paragraph(f"  Maximum: {details.get('max', 'N/A'):.2f}")
            else:
                doc.add_paragraph(f"  Unique Values: {details.get('unique', 'N/A')}")
            doc.add_paragraph("\n")
    except Exception as e:
        doc.add_paragraph(f"Error generating column details: {e}")

    try:
        # Missing values analysis
        doc.add_heading("3. Missing Values Analysis", level=2)
        missing_values = df.isnull().sum()
        total_missing = missing_values.sum()
        doc.add_paragraph(f"Total Missing Values: {total_missing}")
        if total_missing > 0:
            doc.add_paragraph("Columns with Missing Values:")
            for col, miss_count in missing_values[missing_values > 0].items():
                doc.add_paragraph(f"  {col}: {miss_count} missing values")
    except Exception as e:
        doc.add_paragraph(f"Error generating missing values analysis: {e}")

    try:
        # Correlation matrix (only for numeric columns that can be converted to numbers)
        numeric_df = df.select_dtypes(include=["number"])

        # Try to convert non-numeric columns to numeric if possible (e.g., strings like '100' become 100)
        for column in numeric_df.columns:
            numeric_df[column] = pd.to_numeric(numeric_df[column], errors='coerce')
        
        if numeric_df.shape[1] > 1:
            doc.add_heading("4. Correlation Analysis", level=2)
            correlation_matrix = numeric_df.corr()
            doc.add_paragraph("Pairwise correlation matrix:")
            doc.add_paragraph(correlation_matrix.to_string())
        else:
            doc.add_paragraph("No numeric columns available for correlation analysis.")
    except Exception as e:
        doc.add_paragraph(f"Error generating correlation analysis: {e}")

    # Save the Word document
    try:
        doc.save(output_docx_path)
        print(f"EDA report saved as: {output_docx_path}")
    except Exception as e:
        print(f"Error saving the document: {e}")

# Example usage
csv_file_path = "carbon.csv"  # Replace with your CSV file path
generate_eda_report(csv_file_path, "eda_report.docx")

EDA report saved as: eda_report.docx


In [3]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model = "llama3-8b-8192")

In [4]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.vectorstores import VectorStore
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

In [5]:
# !pip install doc2txt

In [6]:
loader = Docx2txtLoader("eda_report.docx")

data = loader.load()

data

[Document(metadata={'source': 'eda_report.docx'}, page_content='Exploratory Data Analysis Report\n\n1. Dataset Summary\n\nNumber of Rows: 5677\n\nNumber of Columns: 5')]

In [7]:
vectorstore = InMemoryVectorStore.from_documents(documents=data,embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))
retriever = vectorstore.as_retriever()

c:\Users\rahul\anaconda3\envs\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain

In [ ]:
# Define the system prompt for the LLM
system_prompt = (
    """You are a highly knowledgeable data scientist, capable of answering any questions in the field of AI, including those related to reports provided by the user.
    Only respond to questions strictly related to AI.
    Ensure all answers are concise, accurate, and explicit."""
    "\n\n"
    "{context}"
)

# Define the prompt template for the conversation
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# Create the question-answer chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)

# Create the retrieval chain with the retriever and question-answer chain
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [10]:
input_data = {
    "input": "Give me the details of the report"
}

response = rag_chain.invoke(input_data)
response['answer']

"I'm happy to help!\n\nAs a doctor, I can provide you with an analysis of the Exploratory Data Analysis (EDA) report. Since the report provided only summary statistics, I'll focus on the available information.\n\nFrom the report, we have the following details:\n\n1. Dataset Summary:\n\t* Number of Rows: 5677\n\t* Number of Columns: 5\n\nThese numbers indicate that the dataset contains 5677 rows (or observations) and 5 columns (or variables) of data. This information provides a basic understanding of the dataset's size and structure.\n\nHowever, to conduct a more comprehensive analysis, I would need additional information about the dataset, such as:\n\n* The type of data in each column (e.g., numerical, categorical, or text)\n* The data distribution, including means, medians, and standard deviations\n* Correlations between variables\n* Any missing values or data quality issues\n\nIf you provide more details about the dataset, I'd be happy to help you with a more in-depth analysis."

In [ ]:
while True:
    input_data = {
        "input": input("You: ")
    }

    response = rag_chain.invoke(input_data)
    print("Bot:", response['answer'])

    if response['answer'] == "Goodbye!":
        break

Bot: This appears to be a report from an exploratory data analysis (EDA) on a dataset. The report provides an initial summary of the dataset, including:

* Number of rows: 5677
* Number of columns: 5

This information provides a basic understanding of the dataset's size and structure. However, to gain a deeper understanding of the data, further analysis and exploration would be necessary. As a doctor, I would typically ask follow-up questions such as: What type of data is being analyzed? What are the variables in each column? What is the distribution of the data? And so on.
Bot: I'm here to help with any medical-related questions or concerns. However, it seems like you've decided to exit our conversation. If you change your mind or have any questions in the future, feel free to reach out to me anytime for assistance. Have a great day!
Bot: I'm here to help with any medical-related questions. Since you've asked to exit, I'll wait for your next question. Please feel free to ask a medical